# 🚀 V7 News Scraper - Optimized Edition

## Changes from V6:
- **Performance**: 8 parallel workers (was 4), 8s timeout (was 12s), 1 retry (was 2)
- **Connection Reuse**: Global requests.Session() for HTTP keep-alive
- **Smart Discovery**: Limited RSS paths (12 → top 8), API paths (8 → top 4)
- **Method Budgets**: Time limits per method prevent runaway timeouts
- **Cross-Section Feeds**: Generic + business/science/energy feeds before tech-specific
- **No Bias**: No early exit, no article cap reduction (preserves completeness)

## Pipeline: RSS → API → HTML → Homepage (Optimized Discovery)

## Target: ~35 min (vs V6 ~50 min) with same article quality

In [1]:
# ============================================================
# CELL 1: IMPORTS & SETUP (V7 OPTIMIZED)
# ============================================================
# All required libraries + global Session for connection reuse

# Standard library
import os
import re
import csv
import time
import logging
import hashlib
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Any
from urllib.parse import urlparse, urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed

# HTTP & Parsing
import requests
from bs4 import BeautifulSoup
import feedparser
from newspaper import Article, Config as NewspaperConfig, build as newspaper_build

# Data handling
import pandas as pd
import numpy as np
from collections import Counter

# Progress display (optional - graceful fallback)
try:
    from tqdm.notebook import tqdm
    TQDM_AVAILABLE = True
except ImportError:
    TQDM_AVAILABLE = False
    print("⚠️ tqdm not installed - using basic progress display")

# ============================================================
# LOGGING SETUP (V7)
# ============================================================
LOG_FILENAME = "scraper_v7.log"
LOG_FORMAT = "%(asctime)s | %(levelname)-8s | %(message)s"
LOG_DATEFMT = "%H:%M:%S"

# Clear existing handlers
logger = logging.getLogger("ScraperV7")
logger.handlers = []
logger.setLevel(logging.INFO)

# File handler
file_handler = logging.FileHandler(LOG_FILENAME, mode='w', encoding='utf-8')
file_handler.setFormatter(logging.Formatter(LOG_FORMAT, datefmt=LOG_DATEFMT))
logger.addHandler(file_handler)

# Console handler (minimal)
console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(message)s"))
console_handler.setLevel(logging.WARNING)
logger.addHandler(console_handler)

# ============================================================
# GLOBAL HTTP SESSION (V7 OPTIMIZATION - Connection Reuse)
# ============================================================
# Single session reuses TCP/TLS connections = ~20-30% faster
HTTP_SESSION = requests.Session()
HTTP_SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
})

# ============================================================
# NEWSPAPER CONFIG (V7 - Faster timeout)
# ============================================================
NEWSPAPER_CONFIG = NewspaperConfig()
NEWSPAPER_CONFIG.browser_user_agent = HTTP_SESSION.headers["User-Agent"]
NEWSPAPER_CONFIG.request_timeout = 10  # Was 15 in V6
NEWSPAPER_CONFIG.fetch_images = False
NEWSPAPER_CONFIG.memoize_articles = False

print("✅ Cell 1 Complete: V7 Optimized imports loaded")
print(f"   📦 Libraries: requests, BeautifulSoup, feedparser, newspaper3k, pandas")
print(f"   📝 Log file: {LOG_FILENAME}")
print(f"   🔌 HTTP Session: Connection reuse enabled (keep-alive)")
print(f"   🎯 tqdm progress: {'Available' if TQDM_AVAILABLE else 'Not available'}")
print(f"   🚀 V7: Optimized edition ready")

✅ Cell 1 Complete: V7 Optimized imports loaded
   📦 Libraries: requests, BeautifulSoup, feedparser, newspaper3k, pandas
   📝 Log file: scraper_v7.log
   🔌 HTTP Session: Connection reuse enabled (keep-alive)
   🎯 tqdm progress: Available
   🚀 V7: Optimized edition ready


In [2]:
# ============================================================
# CELL 2: CONFIGURATION (V7 OPTIMIZED)
# ============================================================
# Keywords, Sources, and OPTIMIZED Scraping Parameters

# ============================================================
# KEYWORDS (Exact terms for production use)
# ============================================================
KEYWORDS = [
    "A.I.",
    "Google",
    "Microsoft",
    "Nvidia",
    "Energy",
    "Chipset"
]

# ============================================================
# V7 OPTIMIZED SCRAPING PARAMETERS
# ============================================================
DAYS_LOOKBACK = 14           # 14 days of news
MAX_ARTICLES_PER_SOURCE = 40 # Keep at 40 (no reduction - we want many articles)
REQUEST_TIMEOUT = 8          # V7: 8s (was 12s) - faster failure detection
MAX_RETRIES = 1              # V7: 1 retry (was 2) - don't waste time on dead endpoints
PARALLEL_WORKERS = 8         # V7: 8 workers (was 4) - 2x parallelism
MIN_CONTENT_LENGTH = 200     # Minimum article length
MIN_HEADLINE_LENGTH = 15     # Minimum headline length

# V7 NEW: Method time budgets (prevent runaway timeouts)
METHOD_TIME_BUDGETS = {
    "rss": 40,       # Max 40s on RSS discovery (was unlimited)
    "api": 25,       # Max 25s on API detection
    "html": 70,      # Max 70s on sitemap parsing
    "homepage": 100  # Max 100s on homepage extraction
}

# V7 NEW: Discovery limits (reduce probe attempts)
MAX_RSS_PATHS = 12       # Try top 12 RSS paths (was 26)
MAX_API_PATHS = 4        # Try top 4 API paths (was 8)
MAX_SITEMAP_PATHS = 8    # Try top 8 sitemap paths (was 15)

# Output
OUTPUT_DIR = "."
OUTPUT_FILENAME = "scraped_news_v7.csv"

# ============================================================
# 50 QUALITY NEWS SOURCES (BASE DOMAINS ONLY - Same as V6)
# ============================================================
SOURCES_CONFIG = [
    # ══════════════════════════════════════════════════════════
    # MAJOR TECH NEWS (15 sources)
    # ══════════════════════════════════════════════════════════
    {"name": "TechCrunch", "url": "https://techcrunch.com"},
    {"name": "The Verge", "url": "https://www.theverge.com"},
    {"name": "Ars Technica", "url": "https://arstechnica.com"},
    {"name": "Wired", "url": "https://www.wired.com"},
    {"name": "Engadget", "url": "https://www.engadget.com"},
    {"name": "CNET", "url": "https://www.cnet.com"},
    {"name": "ZDNet", "url": "https://www.zdnet.com"},
    {"name": "Mashable", "url": "https://mashable.com"},
    {"name": "VentureBeat", "url": "https://venturebeat.com"},
    {"name": "The Next Web", "url": "https://thenextweb.com"},
    {"name": "TechRadar", "url": "https://www.techradar.com"},
    {"name": "Tom's Hardware", "url": "https://www.tomshardware.com"},
    {"name": "AnandTech", "url": "https://www.anandtech.com"},
    {"name": "Gizmodo", "url": "https://gizmodo.com"},
    {"name": "Digital Trends", "url": "https://www.digitaltrends.com"},
    
    # ══════════════════════════════════════════════════════════
    # AI & TECH SPECIALIST (10 sources)
    # ══════════════════════════════════════════════════════════
    {"name": "MIT Technology Review", "url": "https://www.technologyreview.com"},
    {"name": "IEEE Spectrum", "url": "https://spectrum.ieee.org"},
    {"name": "AI Business", "url": "https://aibusiness.com"},
    {"name": "Analytics India Magazine", "url": "https://analyticsindiamag.com"},
    {"name": "The AI Blog", "url": "https://blogs.microsoft.com"},
    {"name": "Google AI Blog", "url": "https://ai.googleblog.com"},
    {"name": "OpenAI Blog", "url": "https://openai.com"},
    {"name": "Synced Review", "url": "https://syncedreview.com"},
    {"name": "Marktechpost", "url": "https://www.marktechpost.com"},
    {"name": "The Decoder", "url": "https://the-decoder.com"},
    
    # ══════════════════════════════════════════════════════════
    # BUSINESS & FINANCE TECH (10 sources)
    # ══════════════════════════════════════════════════════════
    {"name": "Bloomberg Technology", "url": "https://www.bloomberg.com"},
    {"name": "Reuters Technology", "url": "https://www.reuters.com"},
    {"name": "CNBC Tech", "url": "https://www.cnbc.com"},
    {"name": "Financial Times Tech", "url": "https://www.ft.com"},
    {"name": "Wall Street Journal Tech", "url": "https://www.wsj.com"},
    {"name": "Fortune Tech", "url": "https://fortune.com"},
    {"name": "Forbes Tech", "url": "https://www.forbes.com"},
    {"name": "Business Insider Tech", "url": "https://www.businessinsider.com"},
    {"name": "Yahoo Finance Tech", "url": "https://finance.yahoo.com"},
    {"name": "Seeking Alpha Tech", "url": "https://seekingalpha.com"},
    
    # ══════════════════════════════════════════════════════════
    # GENERAL NEWS TECH SECTIONS (10 sources)
    # ══════════════════════════════════════════════════════════
    {"name": "BBC Technology", "url": "https://www.bbc.com"},
    {"name": "CNN Tech", "url": "https://www.cnn.com"},
    {"name": "The Guardian Tech", "url": "https://www.theguardian.com"},
    {"name": "NY Times Tech", "url": "https://www.nytimes.com"},
    {"name": "Washington Post Tech", "url": "https://www.washingtonpost.com"},
    {"name": "NPR Tech", "url": "https://www.npr.org"},
    {"name": "AP Tech", "url": "https://apnews.com"},
    {"name": "Axios Tech", "url": "https://www.axios.com"},
    {"name": "The Atlantic Tech", "url": "https://www.theatlantic.com"},
    {"name": "Politico Tech", "url": "https://www.politico.com"},
    
    # ══════════════════════════════════════════════════════════
    # CHIP & ENERGY SPECIALIST (5 sources)
    # ══════════════════════════════════════════════════════════
    {"name": "SemiAnalysis", "url": "https://www.semianalysis.com"},
    {"name": "EE Times", "url": "https://www.eetimes.com"},
    {"name": "Semiconductor Engineering", "url": "https://semiengineering.com"},
    {"name": "CleanTechnica", "url": "https://cleantechnica.com"},
    {"name": "Electrek", "url": "https://electrek.co"},
]

# ============================================================
# V7 SUMMARY
# ============================================================
print("✅ Cell 2 Complete: V7 Optimized configuration loaded")
print(f"   🔑 Keywords: {len(KEYWORDS)} terms")
print(f"   📰 Sources: {len(SOURCES_CONFIG)} quality news sources")
print(f"   📅 Lookback: {DAYS_LOOKBACK} days")
print(f"   📊 Max articles: {MAX_ARTICLES_PER_SOURCE} per source (unchanged)")
print()
print("   ⚡ V7 Optimizations:")
print(f"      • Workers: {PARALLEL_WORKERS} (was 4)")
print(f"      • Timeout: {REQUEST_TIMEOUT}s (was 12s)")
print(f"      • Retries: {MAX_RETRIES} (was 2)")
print(f"      • RSS paths: {MAX_RSS_PATHS} (was 26)")
print(f"      • API paths: {MAX_API_PATHS} (was 8)")
print(f"      • Method budgets: {METHOD_TIME_BUDGETS}")

✅ Cell 2 Complete: V7 Optimized configuration loaded
   🔑 Keywords: 6 terms
   📰 Sources: 50 quality news sources
   📅 Lookback: 14 days
   📊 Max articles: 40 per source (unchanged)

   ⚡ V7 Optimizations:
      • Workers: 8 (was 4)
      • Timeout: 8s (was 12s)
      • Retries: 1 (was 2)
      • RSS paths: 12 (was 26)
      • API paths: 4 (was 8)
      • Method budgets: {'rss': 40, 'api': 25, 'html': 70, 'homepage': 100}


In [3]:
# ============================================================
# CELL 3: HELPER FUNCTIONS (V7 OPTIMIZED)
# ============================================================
# Core utilities with SESSION-BASED requests for connection reuse

def safe_request(url: str, timeout: int = REQUEST_TIMEOUT, retries: int = MAX_RETRIES) -> Optional[requests.Response]:
    """
    V7 OPTIMIZED: Uses global HTTP_SESSION for connection reuse.
    Make HTTP request with retry logic and error handling.
    Returns None on failure (no blocking on 429/403).
    """
    for attempt in range(retries + 1):
        try:
            # V7: Use session instead of requests.get() - reuses TCP/TLS connections
            response = HTTP_SESSION.get(url, timeout=timeout, allow_redirects=True)
            
            # Quick fail on auth/rate limit - don't retry
            if response.status_code in [401, 403, 429, 451]:
                logger.debug(f"Access denied ({response.status_code}): {url}")
                return None
            
            if response.status_code == 200:
                return response
                
            # Retry on server errors (only if retries left)
            if response.status_code >= 500 and attempt < retries:
                time.sleep(0.5)  # V7: Reduced from 1s
                continue
                
        except requests.exceptions.Timeout:
            logger.debug(f"Timeout (attempt {attempt+1}): {url}")
            if attempt < retries:
                time.sleep(0.3)  # V7: Reduced from 0.5s
                continue
        except requests.exceptions.RequestException as e:
            logger.debug(f"Request error: {e}")
            return None
    
    return None


def parse_date(date_input: Any) -> Optional[datetime]:
    """
    Parse date from various formats (string, struct_time, datetime).
    Returns None if parsing fails.
    """
    if date_input is None:
        return None
    
    # Already datetime
    if isinstance(date_input, datetime):
        return date_input
    
    # feedparser struct_time
    if hasattr(date_input, 'tm_year'):
        try:
            return datetime(*date_input[:6])
        except:
            return None
    
    # String parsing
    if isinstance(date_input, str):
        date_formats = [
            "%Y-%m-%dT%H:%M:%S",
            "%Y-%m-%dT%H:%M:%SZ",
            "%Y-%m-%dT%H:%M:%S%z",
            "%Y-%m-%d %H:%M:%S",
            "%Y-%m-%d",
            "%d %b %Y",
            "%B %d, %Y",
            "%b %d, %Y",
        ]
        
        # Clean timezone info for simpler parsing
        clean_date = re.sub(r'[+-]\d{2}:\d{2}$', '', date_input)
        clean_date = clean_date.replace('Z', '').strip()
        
        for fmt in date_formats:
            try:
                return datetime.strptime(clean_date, fmt)
            except ValueError:
                continue
    
    return None


def is_within_date_range(date_input: Any, days: int = DAYS_LOOKBACK) -> bool:
    """Check if date is within the lookback period."""
    parsed = parse_date(date_input)
    if parsed is None:
        return True  # Include if date unknown
    
    cutoff = datetime.now() - timedelta(days=days)
    return parsed >= cutoff


def generate_url_hash(url: str) -> str:
    """Generate MD5 hash of URL for deduplication."""
    clean_url = url.lower().strip().rstrip('/')
    return hashlib.md5(clean_url.encode()).hexdigest()[:12]


def clean_text(text: Optional[str]) -> str:
    """Clean and normalize text content."""
    if not text:
        return ""
    
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove special characters but keep punctuation
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', text)
    return text.strip()


def extract_domain(url: str) -> str:
    """Extract domain from URL."""
    try:
        parsed = urlparse(url)
        domain = parsed.netloc.lower()
        # Remove www prefix
        if domain.startswith('www.'):
            domain = domain[4:]
        return domain
    except:
        return ""


def matches_keywords(text: str, keywords: List[str] = KEYWORDS) -> Tuple[bool, List[str]]:
    """
    Check if text matches any keywords (FULLY case-insensitive).
    Returns (matched: bool, list of matched keywords).
    
    Matching: "Google" matches "google", "GOOGLE", "GoOgLe", etc.
    Special: "A.I." also matches "AI", "ai", "A.i", etc.
    """
    if not text:
        return False, []
    
    text_lower = text.lower()
    matched = []
    
    for kw in keywords:
        kw_lower = kw.lower()
        
        # Special case: "A.I." should also match "AI" (without dots)
        if kw_lower == "a.i.":
            if "a.i." in text_lower or re.search(r'\bai\b', text_lower):
                matched.append(kw)
        # All other keywords: simple case-insensitive substring match
        else:
            if kw_lower in text_lower:
                matched.append(kw)
    
    return len(matched) > 0, list(set(matched))


def format_elapsed(seconds: float) -> str:
    """Format elapsed time nicely."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        mins = int(seconds // 60)
        secs = int(seconds % 60)
        return f"{mins}m {secs}s"
    else:
        hours = int(seconds // 3600)
        mins = int((seconds % 3600) // 60)
        return f"{hours}h {mins}m"


# ============================================================
# V7 NEW: TIME BUDGET CHECKER
# ============================================================
def check_time_budget(start_time: float, budget: float) -> bool:
    """Check if we've exceeded the time budget for a method."""
    elapsed = time.time() - start_time
    return elapsed < budget


# ============================================================
# PROGRESS DISPLAY HELPER (V7 - Enhanced with ETA)
# ============================================================
class ProgressTracker:
    """Simple progress tracker with live updates and ETA."""
    
    def __init__(self, total: int, description: str = "Progress"):
        self.total = total
        self.current = 0
        self.description = description
        self.start_time = time.time()
        self.results = {"success": 0, "failed": 0, "skipped": 0}
    
    def update(self, status: str = "success", message: str = ""):
        self.current += 1
        self.results[status] = self.results.get(status, 0) + 1
        
        elapsed = time.time() - self.start_time
        pct = (self.current / self.total) * 100
        
        # V7: Calculate ETA
        if self.current > 0:
            avg_time = elapsed / self.current
            remaining = (self.total - self.current) * avg_time
            eta_str = f"ETA {format_elapsed(remaining)}"
        else:
            eta_str = "ETA --"
        
        # Progress bar
        bar_len = 30
        filled = int(bar_len * self.current / self.total)
        bar = "█" * filled + "░" * (bar_len - filled)
        
        status_icon = {"success": "✅", "failed": "❌", "skipped": "⏭️"}.get(status, "•")
        
        print(f"\r[{bar}] {pct:5.1f}% | {self.current}/{self.total} | "
              f"✅{self.results['success']} ❌{self.results['failed']} | "
              f"{format_elapsed(elapsed)} | {eta_str} | {status_icon} {message[:30]:<30}", end="", flush=True)
    
    def finish(self):
        elapsed = time.time() - self.start_time
        print(f"\n{'─'*80}")
        print(f"✅ Complete: {self.results['success']} success, {self.results['failed']} failed, "
              f"{self.results.get('skipped', 0)} skipped in {format_elapsed(elapsed)}")


print("✅ Cell 3 Complete: V7 Optimized helper functions loaded")
print(f"   🔧 safe_request(): Uses HTTP_SESSION for connection reuse")
print(f"   🔧 check_time_budget(): Enforce method time limits")
print(f"   🔧 ProgressTracker: Enhanced with ETA display")
print(f"   🔧 Retry delays reduced: 1s→0.5s, 0.5s→0.3s")

✅ Cell 3 Complete: V7 Optimized helper functions loaded
   🔧 safe_request(): Uses HTTP_SESSION for connection reuse
   🔧 check_time_budget(): Enforce method time limits
   🔧 ProgressTracker: Enhanced with ETA display
   🔧 Retry delays reduced: 1s→0.5s, 0.5s→0.3s


In [4]:
# ============================================================
# CELL 4: QUALITY FILTERS
# ============================================================
# Filter out junk content, ads, navigation, and low-quality articles

# Known junk patterns in headlines
JUNK_HEADLINE_PATTERNS = [
    r'^(video|watch|listen|podcast|gallery|photos?|slideshow)',
    r'^(subscribe|sign up|log ?in|register|newsletter)',
    r'^(ad|sponsored|promoted|partner content)',
    r'^(menu|navigation|home|about|contact)',
    r'^(cookie|privacy|terms|disclaimer)',
    r'(click here|read more|see more|view all)',
    r'^[\d\s\-\/\.]+$',  # Only numbers/dates
    r'^.{0,10}$',  # Too short
]

# Known junk patterns in content
JUNK_CONTENT_PATTERNS = [
    r'subscribe to (our|the) newsletter',
    r'sign up for (our|the|free)',
    r'cookies? (policy|settings|preferences)',
    r'(we use|this site uses) cookies',
    r'javascript (is )?(disabled|required)',
    r'please enable javascript',
    r'your (browser|ad ?blocker)',
]


def is_valid_headline(headline: str) -> Tuple[bool, str]:
    """
    Validate headline quality.
    Returns (is_valid, reason).
    """
    if not headline:
        return False, "empty headline"
    
    headline = headline.strip()
    
    # Length checks
    if len(headline) < MIN_HEADLINE_LENGTH:
        return False, f"too short ({len(headline)} chars)"
    
    if len(headline) > 500:
        return False, "too long (>500 chars)"
    
    # Word count check
    words = headline.split()
    if len(words) < 3:
        return False, f"too few words ({len(words)})"
    
    # Junk pattern check
    headline_lower = headline.lower()
    for pattern in JUNK_HEADLINE_PATTERNS:
        if re.search(pattern, headline_lower):
            return False, f"matches junk pattern"
    
    # All caps check (usually navigation)
    if headline.isupper() and len(headline) > 20:
        return False, "all caps"
    
    return True, "valid"


def is_valid_content(content: str) -> Tuple[bool, str]:
    """
    Validate content quality.
    Returns (is_valid, reason).
    """
    if not content:
        return False, "empty content"
    
    content = content.strip()
    
    # Length check
    if len(content) < MIN_CONTENT_LENGTH:
        return False, f"too short ({len(content)} chars)"
    
    # Word count
    words = content.split()
    if len(words) < 30:
        return False, f"too few words ({len(words)})"
    
    # Sentence check (must have at least one sentence)
    if not re.search(r'[.!?]', content):
        return False, "no sentences"
    
    # Junk pattern check
    content_lower = content.lower()
    junk_score = 0
    for pattern in JUNK_CONTENT_PATTERNS:
        if re.search(pattern, content_lower):
            junk_score += 1
    
    if junk_score >= 2:
        return False, f"too many junk patterns ({junk_score})"
    
    # Check for actual article content (not just navigation)
    sentence_count = len(re.findall(r'[.!?]+', content))
    if sentence_count < 2:
        return False, f"too few sentences ({sentence_count})"
    
    return True, "valid"


def is_quality_article(article: Dict) -> Tuple[bool, str]:
    """
    Combined quality check for an article.
    Returns (is_quality, reason).
    """
    # Check headline
    headline = article.get('headline', '') or article.get('title', '')
    valid_headline, headline_reason = is_valid_headline(headline)
    if not valid_headline:
        return False, f"headline: {headline_reason}"
    
    # Check content
    content = article.get('full_content', '') or article.get('content', '')
    valid_content, content_reason = is_valid_content(content)
    if not valid_content:
        return False, f"content: {content_reason}"
    
    # Check URL exists
    url = article.get('url', '')
    if not url or not url.startswith('http'):
        return False, "invalid URL"
    
    # Date check (within lookback period)
    pub_date = article.get('published', '') or article.get('date', '')
    if pub_date and not is_within_date_range(pub_date):
        return False, "too old"
    
    return True, "quality article"


def filter_quality_articles(articles: List[Dict], source_name: str = "") -> List[Dict]:
    """
    Filter a list of articles for quality.
    Returns only quality articles with rejection stats.
    """
    quality_articles = []
    rejection_reasons = Counter()
    
    for article in articles:
        is_quality, reason = is_quality_article(article)
        if is_quality:
            quality_articles.append(article)
        else:
            rejection_reasons[reason] += 1
    
    # Log rejection summary
    if rejection_reasons:
        logger.info(f"{source_name}: Rejected {sum(rejection_reasons.values())} articles - "
                   f"Top reasons: {dict(rejection_reasons.most_common(3))}")
    
    return quality_articles


print("✅ Cell 4 Complete: Quality filters loaded")
print(f"   🔍 Headline validation: length, word count, junk patterns")
print(f"   🔍 Content validation: length, sentences, junk patterns")
print(f"   🔍 Article validation: combined headline + content + URL + date")
print(f"   📊 Rejection tracking with detailed reasons")

✅ Cell 4 Complete: Quality filters loaded
   🔍 Headline validation: length, word count, junk patterns
   🔍 Content validation: length, sentences, junk patterns
   🔍 Article validation: combined headline + content + URL + date
   📊 Rejection tracking with detailed reasons


In [5]:
# ============================================================
# CELL 5: SCRAPER METHODS (V7 OPTIMIZED)
# ============================================================
# Four scraping methods with OPTIMIZED discovery:
# - Reduced path attempts (faster failure)
# - Cross-section feeds (broader keyword coverage)
# - Time budget enforcement
# - Session-based requests

# ============================================================
# TECH & CROSS-SECTION URL PATTERNS (V7: Broader coverage)
# ============================================================
TECH_URL_PATTERNS = [
    '/tech', '/technology', '/ai/', '/artificial-intelligence',
    '/machine-learning', '/ml/', '/computing', '/software',
    '/hardware', '/chips', '/semiconductor', '/nvidia', '/google',
    '/microsoft', '/apple', '/amazon', '/meta/', '/openai',
    '/digital', '/cyber', '/data', '/cloud', '/startup',
    '/innovation', '/science', '/energy', '/electric', '/ev/',
    '/business/tech', '/news/tech', '/section/tech',
    # V7: Added cross-section patterns for Energy/Chipset coverage
    '/business', '/markets', '/finance', '/economy',
    '/environment', '/climate', '/sustainability'
]

def is_tech_url(url: str) -> bool:
    """Check if URL likely contains relevant content (tech + cross-section)."""
    url_lower = url.lower()
    return any(pattern in url_lower for pattern in TECH_URL_PATTERNS)


def fetch_full_article(url: str) -> Dict:
    """
    Download and parse a single article URL using newspaper3k.
    Returns article dict with full content.
    """
    try:
        article = Article(url, config=NEWSPAPER_CONFIG)
        article.download()
        article.parse()
        
        return {
            "headline": clean_text(article.title),
            "author": ", ".join(article.authors) if article.authors else "",
            "url": url,
            "published": article.publish_date.isoformat() if article.publish_date else "",
            "full_content": clean_text(article.text),
            "content_snippet": clean_text(article.text[:300]) if article.text else "",
        }
    except Exception as e:
        logger.debug(f"Failed to fetch article {url}: {e}")
        return None


def fetch_articles_parallel(urls: List[str], max_workers: int = PARALLEL_WORKERS) -> List[Dict]:
    """
    V7: Fetch multiple articles in parallel with increased workers.
    Returns list of successfully parsed articles.
    """
    articles = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_full_article, url): url for url in urls}
        
        for future in as_completed(futures):
            result = future.result()
            if result and result.get('headline'):
                articles.append(result)
    
    return articles


# ============================================================
# V7 RSS PATH ORDERING (Cross-section first, then tech-specific)
# ============================================================
# V7 STRATEGY: Generic feeds first (broader coverage), then cross-section 
# for Energy/Business, then tech-specific last
RSS_PATHS_V7 = [
    # Generic feeds first (broadest coverage)
    "/feed", "/rss", "/rss.xml", "/feed.xml",
    # Cross-section feeds (for Energy, Business, Markets keywords)
    "/business/feed", "/business/rss",
    "/science/feed", "/energy/feed", "/markets/feed",
    # Tech-specific paths (checked last, limited to top 4)
    "/technology/feed", "/tech/feed", "/ai/feed",
    "/feeds/posts/default"  # Blogger/generic
][:MAX_RSS_PATHS]  # V7: Limit to MAX_RSS_PATHS (12)

# V7 API paths (reduced from 8 to 4)
API_PATHS_V7 = [
    "/wp-json/wp/v2/posts",    # WordPress (most common)
    "/api/posts",              # Generic
    "/api/v1/posts",           # Versioned
    "/api/articles"            # Alternative
][:MAX_API_PATHS]  # V7: Limit to MAX_API_PATHS (4)

# V7 Sitemap paths (reduced from 15 to 8)
SITEMAP_PATHS_V7 = [
    "/news-sitemap.xml", "/sitemap-news.xml",  # News-specific first
    "/post-sitemap.xml", "/article-sitemap.xml",
    "/sitemap.xml", "/sitemap_index.xml",      # General fallback
    "/sitemap-posts.xml", "/blog-sitemap.xml"
][:MAX_SITEMAP_PATHS]


# ============================================================
# METHOD 1: RSS FEED (V7 OPTIMIZED)
# ============================================================
def find_rss_feed_v7(base_url: str, time_budget: float) -> Optional[str]:
    """
    V7 OPTIMIZED: Find RSS feed with time budget and reduced paths.
    Uses cross-section feeds for broader keyword coverage.
    """
    start_time = time.time()
    
    # V7: Try RSS paths with time budget
    for path in RSS_PATHS_V7:
        if not check_time_budget(start_time, time_budget):
            logger.debug(f"RSS discovery timed out after {time_budget}s")
            break
            
        feed_url = urljoin(base_url, path)
        response = safe_request(feed_url, timeout=REQUEST_TIMEOUT)
        if response and ('xml' in response.headers.get('content-type', '') or 
                        '<rss' in response.text[:500] or '<feed' in response.text[:500]):
            logger.debug(f"Found RSS feed: {feed_url}")
            return feed_url
    
    # V7: Try HTML <link> tags only if we have time budget left
    if check_time_budget(start_time, time_budget):
        response = safe_request(base_url, timeout=REQUEST_TIMEOUT)
        if response:
            soup = BeautifulSoup(response.text, 'html.parser')
            for link in soup.find_all('link', type=re.compile(r'(rss|atom|xml)')):
                href = link.get('href')
                if href:
                    return urljoin(base_url, href)
    
    return None


def scrape_rss(source: Dict, time_budget: float = None) -> Dict:
    """
    V7 OPTIMIZED: Scrape articles from RSS feed with time budget.
    """
    source_name = source['name']
    base_url = source['url']
    start_time = time.time()
    budget = time_budget or METHOD_TIME_BUDGETS["rss"]
    
    result = {
        "source": source_name,
        "method": "rss",
        "articles": [],
        "success": False,
        "error": None,
        "elapsed": 0
    }
    
    try:
        # V7: Find RSS feed with time budget
        feed_url = find_rss_feed_v7(base_url, budget * 0.4)  # 40% budget for discovery
        if not feed_url:
            result["error"] = "No RSS feed found"
            result["elapsed"] = time.time() - start_time
            return result
        
        # Parse feed
        feed = feedparser.parse(feed_url)
        if not feed.entries:
            result["error"] = "Empty RSS feed"
            result["elapsed"] = time.time() - start_time
            return result
        
        # Collect article URLs (within date range)
        article_urls = []
        for entry in feed.entries[:MAX_ARTICLES_PER_SOURCE * 2]:
            pub_date = entry.get('published_parsed') or entry.get('updated_parsed')
            if pub_date and not is_within_date_range(pub_date):
                continue
            
            url = entry.get('link', '')
            if url and url.startswith('http'):
                article_urls.append(url)
        
        if not article_urls:
            result["error"] = "No recent articles in RSS"
            result["elapsed"] = time.time() - start_time
            return result
        
        # V7: Fetch full articles with increased parallelism
        articles = fetch_articles_parallel(article_urls[:MAX_ARTICLES_PER_SOURCE])
        
        for art in articles:
            art['source'] = source_name
            art['method'] = 'rss'
            art['url_hash'] = generate_url_hash(art['url'])
        
        result["articles"] = articles
        result["success"] = len(articles) > 0
        
    except Exception as e:
        result["error"] = str(e)
        logger.error(f"RSS error for {source_name}: {e}")
    
    result["elapsed"] = time.time() - start_time
    return result


# ============================================================
# METHOD 2: API (V7 OPTIMIZED - Reduced paths)
# ============================================================
def scrape_api(source: Dict, time_budget: float = None) -> Dict:
    """
    V7 OPTIMIZED: Scrape from WordPress/JSON API with reduced paths.
    """
    source_name = source['name']
    base_url = source['url']
    start_time = time.time()
    budget = time_budget or METHOD_TIME_BUDGETS["api"]
    
    result = {
        "source": source_name,
        "method": "api",
        "articles": [],
        "success": False,
        "error": None,
        "elapsed": 0
    }
    
    try:
        for api_path in API_PATHS_V7:
            # V7: Check time budget
            if not check_time_budget(start_time, budget):
                result["error"] = "API discovery timed out"
                break
                
            api_url = urljoin(base_url, api_path)
            params = f"?per_page={min(MAX_ARTICLES_PER_SOURCE, 100)}&_embed"
            
            response = safe_request(api_url + params, timeout=REQUEST_TIMEOUT)
            if not response:
                continue
            
            try:
                posts = response.json()
            except:
                continue
            
            if not isinstance(posts, list) or len(posts) == 0:
                continue
            
            # Parse API response
            articles = []
            for post in posts[:MAX_ARTICLES_PER_SOURCE]:
                try:
                    pub_date = post.get('date', '') or post.get('published', '') or post.get('publishedAt', '')
                    if pub_date and not is_within_date_range(pub_date):
                        continue
                    
                    title = post.get('title', {})
                    if isinstance(title, dict):
                        title = title.get('rendered', '')
                    
                    content = post.get('content', {})
                    if isinstance(content, dict):
                        content = content.get('rendered', '')
                    
                    excerpt = post.get('excerpt', {})
                    if isinstance(excerpt, dict):
                        excerpt = excerpt.get('rendered', '')
                    
                    url = post.get('link', '') or post.get('url', '') or post.get('href', '')
                    
                    if not title or not url:
                        continue
                    
                    soup = BeautifulSoup(str(content), 'html.parser')
                    clean_content = soup.get_text(separator=' ', strip=True)
                    
                    soup_excerpt = BeautifulSoup(str(excerpt), 'html.parser')
                    clean_excerpt = soup_excerpt.get_text(separator=' ', strip=True)
                    
                    author = ""
                    if '_embedded' in post and 'author' in post['_embedded']:
                        authors = post['_embedded']['author']
                        if authors:
                            author = authors[0].get('name', '')
                    elif 'author' in post:
                        auth = post['author']
                        if isinstance(auth, dict):
                            author = auth.get('name', '') or auth.get('displayName', '')
                        elif isinstance(auth, str):
                            author = auth
                    
                    articles.append({
                        "source": source_name,
                        "method": "api",
                        "headline": clean_text(str(title)),
                        "author": author,
                        "url": url,
                        "published": pub_date,
                        "full_content": clean_text(clean_content),
                        "content_snippet": clean_text(clean_excerpt[:300]),
                        "url_hash": generate_url_hash(url)
                    })
                    
                except Exception as e:
                    logger.debug(f"API parse error: {e}")
                    continue
            
            if articles:
                result["articles"] = articles
                result["success"] = True
                break
        
        if not result["success"]:
            result["error"] = result.get("error") or "No working API found"
        
    except Exception as e:
        result["error"] = str(e)
        logger.error(f"API error for {source_name}: {e}")
    
    result["elapsed"] = time.time() - start_time
    return result


# ============================================================
# METHOD 3: SITEMAP (V7 OPTIMIZED - Reduced paths)
# ============================================================
def find_sitemap_v7(base_url: str, time_budget: float) -> Optional[str]:
    """
    V7 OPTIMIZED: Find sitemap with time budget and reduced paths.
    """
    start_time = time.time()
    
    # V7: Check robots.txt first (quick check)
    robots_url = urljoin(base_url, "/robots.txt")
    response = safe_request(robots_url, timeout=5)
    if response:
        for line in response.text.split('\n'):
            if line.lower().startswith('sitemap:'):
                sitemap_url = line.split(':', 1)[1].strip()
                if 'news' in sitemap_url.lower() or 'post' in sitemap_url.lower():
                    return sitemap_url
                # Store first sitemap as fallback
                if not hasattr(find_sitemap_v7, '_fallback'):
                    find_sitemap_v7._fallback = sitemap_url
    
    # V7: Try sitemap paths with time budget
    for path in SITEMAP_PATHS_V7:
        if not check_time_budget(start_time, time_budget):
            break
            
        sitemap_url = urljoin(base_url, path)
        response = safe_request(sitemap_url, timeout=5)
        if response and ('<urlset' in response.text[:1000] or '<sitemapindex' in response.text[:1000]):
            return sitemap_url
    
    # Use fallback from robots.txt
    if hasattr(find_sitemap_v7, '_fallback'):
        fallback = find_sitemap_v7._fallback
        del find_sitemap_v7._fallback
        return fallback
    
    return None


def scrape_html(source: Dict, time_budget: float = None) -> Dict:
    """
    V7 OPTIMIZED: Scrape from sitemap with time budget.
    """
    source_name = source['name']
    base_url = source['url']
    start_time = time.time()
    budget = time_budget or METHOD_TIME_BUDGETS["html"]
    
    result = {
        "source": source_name,
        "method": "html",
        "articles": [],
        "success": False,
        "error": None,
        "elapsed": 0
    }
    
    try:
        # V7: Find sitemap with time budget
        sitemap_url = find_sitemap_v7(base_url, budget * 0.3)
        if not sitemap_url:
            result["error"] = "No sitemap found"
            result["elapsed"] = time.time() - start_time
            return result
        
        response = safe_request(sitemap_url, timeout=10)
        if not response:
            result["error"] = "Sitemap not accessible"
            result["elapsed"] = time.time() - start_time
            return result
        
        soup = BeautifulSoup(response.text, 'xml')
        urls = soup.find_all('url')
        
        if not urls:
            result["error"] = "No URLs in sitemap"
            result["elapsed"] = time.time() - start_time
            return result
        
        # Separate tech URLs from general URLs
        tech_urls = []
        general_urls = []
        
        for url_tag in urls:
            loc = url_tag.find('loc')
            lastmod = url_tag.find('lastmod')
            
            if not loc:
                continue
            
            url = loc.text.strip()
            
            if any(skip in url for skip in ['/tag/', '/category/', '/author/', '/page/', '/wp-content/', '/wp-admin/']):
                continue
            
            if lastmod:
                if not is_within_date_range(lastmod.text):
                    continue
            
            if is_tech_url(url):
                tech_urls.append(url)
            else:
                general_urls.append(url)
        
        article_urls = tech_urls[:MAX_ARTICLES_PER_SOURCE] + general_urls[:MAX_ARTICLES_PER_SOURCE - len(tech_urls)]
        
        if not article_urls:
            result["error"] = "No recent articles in sitemap"
            result["elapsed"] = time.time() - start_time
            return result
        
        articles = fetch_articles_parallel(article_urls[:MAX_ARTICLES_PER_SOURCE])
        
        for art in articles:
            art['source'] = source_name
            art['method'] = 'html'
            art['url_hash'] = generate_url_hash(art['url'])
        
        result["articles"] = articles
        result["success"] = len(articles) > 0
        
    except Exception as e:
        result["error"] = str(e)
        logger.error(f"HTML error for {source_name}: {e}")
    
    result["elapsed"] = time.time() - start_time
    return result


# ============================================================
# METHOD 4: HOMEPAGE (V7 OPTIMIZED - Time budget)
# ============================================================
def scrape_homepage(source: Dict, time_budget: float = None) -> Dict:
    """
    V7 OPTIMIZED: Scrape from homepage with time budget.
    """
    source_name = source['name']
    base_url = source['url']
    start_time = time.time()
    budget = time_budget or METHOD_TIME_BUDGETS["homepage"]
    
    result = {
        "source": source_name,
        "method": "homepage",
        "articles": [],
        "success": False,
        "error": None,
        "elapsed": 0
    }
    
    try:
        paper = newspaper_build(base_url, config=NEWSPAPER_CONFIG, memoize_articles=False)
        
        if not paper.articles:
            result["error"] = "No articles detected on homepage"
            result["elapsed"] = time.time() - start_time
            return result
        
        tech_urls = []
        general_urls = []
        
        for art in paper.articles:
            if art.url:
                if is_tech_url(art.url):
                    tech_urls.append(art.url)
                else:
                    general_urls.append(art.url)
        
        article_urls = tech_urls[:MAX_ARTICLES_PER_SOURCE * 2] + general_urls[:MAX_ARTICLES_PER_SOURCE]
        
        if not article_urls:
            result["error"] = "No valid article URLs found"
            result["elapsed"] = time.time() - start_time
            return result
        
        articles = fetch_articles_parallel(article_urls[:MAX_ARTICLES_PER_SOURCE])
        
        for art in articles:
            art['source'] = source_name
            art['method'] = 'homepage'
            art['url_hash'] = generate_url_hash(art['url'])
        
        result["articles"] = articles
        result["success"] = len(articles) > 0
        
    except Exception as e:
        result["error"] = str(e)
        logger.error(f"Homepage error for {source_name}: {e}")
    
    result["elapsed"] = time.time() - start_time
    return result


print("✅ Cell 5 Complete: V7 OPTIMIZED Scraper methods loaded")
print(f"   📡 RSS: {len(RSS_PATHS_V7)} paths (was 26) + cross-section feeds")
print(f"   🔌 API: {len(API_PATHS_V7)} paths (was 8)")
print(f"   🌐 HTML: {len(SITEMAP_PATHS_V7)} sitemap paths (was 15)")
print(f"   🏠 Homepage: With time budget enforcement")
print(f"   ⚡ Parallel workers: {PARALLEL_WORKERS} (was 4)")
print()
print("   🎯 V7 Improvements:")
print("      • Cross-section feeds: /business/feed, /science/feed, /energy/feed")
print("      • Time budgets: Prevent runaway timeouts")
print("      • Reduced discovery: Faster failure on dead endpoints")
print("      • Session reuse: Connection keep-alive")

✅ Cell 5 Complete: V7 OPTIMIZED Scraper methods loaded
   📡 RSS: 12 paths (was 26) + cross-section feeds
   🔌 API: 4 paths (was 8)
   🌐 HTML: 8 sitemap paths (was 15)
   🏠 Homepage: With time budget enforcement
   ⚡ Parallel workers: 8 (was 4)

   🎯 V7 Improvements:
      • Cross-section feeds: /business/feed, /science/feed, /energy/feed
      • Time budgets: Prevent runaway timeouts
      • Reduced discovery: Faster failure on dead endpoints
      • Session reuse: Connection keep-alive


In [6]:
# ============================================================
# CELL 6: WATERFALL LOGIC (V7 OPTIMIZED - Time Budgets)
# ============================================================
# Try methods in order with TIME BUDGETS, STOP as soon as one works

# FIXED ORDER for all sources (fast to slow)
METHOD_ORDER = ["rss", "api", "html", "homepage"]

# Method functions mapping
METHOD_FUNCTIONS = {
    "rss": scrape_rss,
    "api": scrape_api,
    "html": scrape_html,
    "homepage": scrape_homepage,
}


def scrape_source_waterfall(source: Dict) -> Dict:
    """
    V7 OPTIMIZED: Scrape a source using waterfall with TIME BUDGETS.
    Tries methods in order: RSS → API → HTML → Homepage
    STOPS IMMEDIATELY when ANY method returns keyword-matched articles.
    Each method has its own time budget to prevent runaway timeouts.
    
    Returns:
        Dict with results and telemetry
    """
    source_name = source['name']
    start_time = time.time()
    
    result = {
        "source": source_name,
        "url": source['url'],
        "articles": [],
        "method_used": None,
        "methods_tried": [],
        "methods_results": {},
        "total_elapsed": 0,
        "success": False,
    }
    
    # Try each method in order with time budgets, STOP on first success
    for method in METHOD_ORDER:
        method_func = METHOD_FUNCTIONS[method]
        method_budget = METHOD_TIME_BUDGETS[method]
        
        try:
            # V7: Pass time budget to method
            method_result = method_func(source, time_budget=method_budget)
            result["methods_tried"].append(method)
            result["methods_results"][method] = {
                "success": method_result["success"],
                "articles": len(method_result.get("articles", [])),
                "error": method_result.get("error"),
                "elapsed": method_result.get("elapsed", 0),
                "budget": method_budget  # V7: Track budget
            }
            
            if method_result["success"] and method_result["articles"]:
                articles = method_result["articles"]
                
                # Apply quality filter
                quality_articles = filter_quality_articles(articles, source_name)
                
                # Apply keyword filter
                keyword_matched = []
                for art in quality_articles:
                    text_to_check = f"{art.get('headline', '')} {art.get('full_content', '')}"
                    matched, keywords = matches_keywords(text_to_check)
                    if matched:
                        art['matched_keywords'] = ", ".join(keywords[:5])
                        keyword_matched.append(art)
                
                # SUCCESS: Got keyword-matched articles - STOP HERE
                if keyword_matched:
                    result["articles"] = keyword_matched[:MAX_ARTICLES_PER_SOURCE]
                    result["method_used"] = method
                    result["success"] = True
                    logger.info(f"{source_name}: ✅ Got {len(keyword_matched)} articles from {method}, DONE")
                    break  # <-- STOP! Don't try other methods
                else:
                    logger.debug(f"{source_name}: {method} found articles but none matched keywords, trying next...")
            
        except Exception as e:
            logger.error(f"{source_name}: {method} error - {e}")
            result["methods_results"][method] = {
                "success": False,
                "articles": 0,
                "error": str(e),
                "elapsed": 0,
                "budget": method_budget
            }
    
    result["total_elapsed"] = time.time() - start_time
    return result


def get_method_icon(method: str) -> str:
    """Get emoji icon for method."""
    return {"rss": "📡", "api": "🔌", "html": "🌐", "homepage": "🏠"}.get(method, "•")


def format_source_result(result: Dict) -> str:
    """Format source result for display."""
    source = result["source"]
    articles = len(result["articles"])
    method = result["method_used"]
    elapsed = result["total_elapsed"]
    
    if result["success"]:
        icon = get_method_icon(method)
        return f"✅ {source}: {articles} articles via {icon} {method} ({elapsed:.1f}s)"
    else:
        errors = [f"{m}: {r['error']}" for m, r in result["methods_results"].items() if r.get('error')]
        error_str = errors[0] if errors else "unknown"
        return f"❌ {source}: Failed - {error_str[:50]} ({elapsed:.1f}s)"


print("✅ Cell 6 Complete: V7 Waterfall logic with TIME BUDGETS")
print(f"   🔄 Order: 📡RSS → 🔌API → 🌐HTML → 🏠Homepage")
print(f"   ⏱️ Time budgets: RSS={METHOD_TIME_BUDGETS['rss']}s, API={METHOD_TIME_BUDGETS['api']}s, HTML={METHOD_TIME_BUDGETS['html']}s, Homepage={METHOD_TIME_BUDGETS['homepage']}s")
print(f"   🛑 STOP immediately when keyword-matched articles found")
print(f"   🔍 Quality + Keyword filtering integrated")
print()
print("   🎯 V7 Improvements:")
print("      • Time budgets prevent runaway timeouts")
print("      • Budget tracking in telemetry")
print("      • Faster failure = faster overall execution")

✅ Cell 6 Complete: V7 Waterfall logic with TIME BUDGETS
   🔄 Order: 📡RSS → 🔌API → 🌐HTML → 🏠Homepage
   ⏱️ Time budgets: RSS=40s, API=25s, HTML=70s, Homepage=100s
   🛑 STOP immediately when keyword-matched articles found
   🔍 Quality + Keyword filtering integrated

   🎯 V7 Improvements:
      • Time budgets prevent runaway timeouts
      • Budget tracking in telemetry
      • Faster failure = faster overall execution


In [7]:
# ============================================================
# CELL 7: MAIN EXECUTION LOOP (V7 OPTIMIZED)
# ============================================================
# Scrape all 50 sources with V7 optimizations and progress tracking

print("=" * 70)
print("🚀 V7 NEWS SCRAPER - OPTIMIZED EDITION - STARTING")
print("=" * 70)
print(f"📰 Sources: {len(SOURCES_CONFIG)} (base domains only)")
print(f"🔑 Keywords: {KEYWORDS}")
print(f"📅 Lookback: {DAYS_LOOKBACK} days")
print(f"📊 Max articles per source: {MAX_ARTICLES_PER_SOURCE}")
print(f"🔄 Methods: RSS → API → HTML → Homepage (with time budgets)")
print()
print("⚡ V7 Optimizations Active:")
print(f"   • Workers: {PARALLEL_WORKERS} (2x parallelism)")
print(f"   • Timeout: {REQUEST_TIMEOUT}s (33% faster failure)")
print(f"   • Retries: {MAX_RETRIES} (50% less retry time)")
print(f"   • RSS paths: {MAX_RSS_PATHS} (54% fewer probes)")
print(f"   • API paths: {MAX_API_PATHS} (50% fewer probes)")
print(f"   • Session reuse: HTTP keep-alive enabled")
print("=" * 70)
print()

# Initialize tracking
all_results = []
all_articles = []
start_time = time.time()

# Progress tracker with ETA
progress = ProgressTracker(len(SOURCES_CONFIG), "Scraping sources")

# Scrape each source
for idx, source in enumerate(SOURCES_CONFIG, 1):
    source_name = source['name']
    
    try:
        # Scrape with optimized waterfall
        result = scrape_source_waterfall(source)
        all_results.append(result)
        
        # Collect articles
        if result["articles"]:
            all_articles.extend(result["articles"])
            status = "success"
            message = f"{source_name} ({len(result['articles'])} articles)"
        else:
            status = "failed"
            message = f"{source_name} (no articles)"
        
        progress.update(status, message)
        
    except Exception as e:
        logger.error(f"Fatal error for {source_name}: {e}")
        progress.update("failed", f"{source_name} (error)")
        all_results.append({
            "source": source_name,
            "url": source['url'],
            "articles": [],
            "method_used": None,
            "methods_tried": [],
            "methods_results": {},
            "total_elapsed": 0,
            "success": False,
        })

# Finish progress
progress.finish()

# Calculate stats
total_time = time.time() - start_time
successful_sources = sum(1 for r in all_results if r["success"])
total_articles = len(all_articles)

# Summary
print()
print("=" * 70)
print("📊 V7 SCRAPING COMPLETE - SUMMARY")
print("=" * 70)
print(f"⏱️  Total time: {format_elapsed(total_time)}")
print(f"📰 Sources scraped: {successful_sources}/{len(SOURCES_CONFIG)} ({100*successful_sources/len(SOURCES_CONFIG):.1f}%)")
print(f"📝 Total articles: {total_articles}")
print(f"⚡ Articles per minute: {60 * total_articles / total_time:.1f}")
print()

# Method breakdown
method_counts = Counter(r["method_used"] for r in all_results if r["method_used"])
print("📡 Articles by method:")
for method, count in method_counts.most_common():
    icon = get_method_icon(method)
    articles_by_method = sum(len(r["articles"]) for r in all_results if r["method_used"] == method)
    print(f"   {icon} {method}: {count} sources, {articles_by_method} articles")

# V7: Performance comparison estimate
print()
print("📈 V7 Performance Estimate:")
v6_estimated = 50  # minutes
v7_actual = total_time / 60
speedup = v6_estimated / v7_actual if v7_actual > 0 else 0
print(f"   V6 baseline: ~50 min")
print(f"   V7 actual: {v7_actual:.1f} min")
print(f"   Speedup: {speedup:.1f}x")

print()
print("=" * 70)

🚀 V7 NEWS SCRAPER - OPTIMIZED EDITION - STARTING
📰 Sources: 50 (base domains only)
🔑 Keywords: ['A.I.', 'Google', 'Microsoft', 'Nvidia', 'Energy', 'Chipset']
📅 Lookback: 14 days
📊 Max articles per source: 40
🔄 Methods: RSS → API → HTML → Homepage (with time budgets)

⚡ V7 Optimizations Active:
   • Workers: 8 (2x parallelism)
   • Timeout: 8s (33% faster failure)
   • Retries: 1 (50% less retry time)
   • RSS paths: 12 (54% fewer probes)
   • API paths: 4 (50% fewer probes)
   • Session reuse: HTTP keep-alive enabled

[██████████████████████████████] 100.0% | 50/50 | ✅36 ❌14 | 38m 19s | ETA 0.0s | ✅ Electrek (22 articles)        a 
────────────────────────────────────────────────────────────────────────────────
✅ Complete: 36 success, 14 failed, 0 skipped in 38m 19s

📊 V7 SCRAPING COMPLETE - SUMMARY
⏱️  Total time: 38m 19s
📰 Sources scraped: 36/50 (72.0%)
📝 Total articles: 394
⚡ Articles per minute: 10.3

📡 Articles by method:
   📡 rss: 23 sources, 249 articles
   🌐 html: 9 sources, 91

In [8]:
# ============================================================
# CELL 8: DATA EXPORT & CLEANING (V7)
# ============================================================
# Export scraped articles to CSV with proper schema

# Define output schema (11 columns)
OUTPUT_SCHEMA = [
    "source",           # News source name
    "headline",         # Article headline/title
    "author",           # Author name(s)
    "url",              # Article URL
    "published",        # Publication date
    "matched_keywords", # Keywords that matched
    "content_snippet",  # First 300 chars preview
    "url_hash",         # MD5 hash for deduplication
    "full_content",     # Full article text
    "method",           # Scraping method used
    "scraped_at"        # Timestamp of scraping
]

def prepare_article_for_export(article: Dict) -> Dict:
    """Prepare article dict with all required fields."""
    return {
        "source": article.get("source", ""),
        "headline": article.get("headline", ""),
        "author": article.get("author", ""),
        "url": article.get("url", ""),
        "published": article.get("published", ""),
        "matched_keywords": article.get("matched_keywords", ""),
        "content_snippet": article.get("content_snippet", "")[:300],
        "url_hash": article.get("url_hash", generate_url_hash(article.get("url", ""))),
        "full_content": article.get("full_content", ""),
        "method": article.get("method", ""),
        "scraped_at": datetime.now().isoformat()
    }

def export_to_csv(articles: List[Dict], filename: str) -> int:
    """Export articles to CSV file."""
    if not articles:
        print("⚠️ No articles to export")
        return 0
    
    filepath = os.path.join(OUTPUT_DIR, filename)
    
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_SCHEMA)
        writer.writeheader()
        
        for article in articles:
            row = {k: article.get(k, "") for k in OUTPUT_SCHEMA}
            writer.writerow(row)
    
    print(f"✅ Exported {len(articles)} articles to {filepath}")
    return len(articles)

# Prepare articles for export
print("📦 Preparing articles for export...")
export_articles = []

for result in all_results:
    for article in result.get("articles", []):
        export_articles.append(prepare_article_for_export(article))

print(f"   Total articles: {len(export_articles)}")

# V7: Export main CSV with V7 filename
exported_count = export_to_csv(export_articles, "scraped_news_v7.csv")

# V7: Export telemetry with V7 filename
telemetry_filename = "scraping_telemetry_v7.csv"
telemetry_schema = ["source", "success", "method_used", "articles_count", 
                    "methods_tried", "elapsed_seconds", "url"]

with open(telemetry_filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=telemetry_schema)
    writer.writeheader()
    
    for result in all_results:
        writer.writerow({
            "source": result["source"],
            "success": result["success"],
            "method_used": result["method_used"] or "",
            "articles_count": len(result["articles"]),
            "methods_tried": " → ".join(result["methods_tried"]),
            "elapsed_seconds": f"{result['total_elapsed']:.1f}",
            "url": result["url"]
        })

print(f"✅ Exported telemetry to {telemetry_filename}")
print()
print(f"📁 V7 Output files:")
print(f"   • scraped_news_v7.csv ({exported_count} articles)")
print(f"   • {telemetry_filename} ({len(all_results)} sources)")

📦 Preparing articles for export...
   Total articles: 394
✅ Exported 394 articles to .\scraped_news_v7.csv
✅ Exported telemetry to scraping_telemetry_v7.csv

📁 V7 Output files:
   • scraped_news_v7.csv (394 articles)
   • scraping_telemetry_v7.csv (50 sources)


In [9]:
# ============================================================
# CELL 9: ANALYTICS DASHBOARD
# ============================================================
# Detailed analysis of scraped data

print("=" * 70)
print("📊 ANALYTICS DASHBOARD")
print("=" * 70)
print()

# Load data into DataFrame
df = pd.DataFrame(export_articles)

if len(df) == 0:
    print("⚠️ No data to analyze")
else:
    # ─────────────────────────────────────────────────────────
    # 1. BASIC STATS
    # ─────────────────────────────────────────────────────────
    print("📈 BASIC STATISTICS")
    print("─" * 50)
    print(f"   Total articles: {len(df)}")
    print(f"   Unique sources: {df['source'].nunique()}")
    print(f"   Date range: {df['published'].min()} to {df['published'].max()}")
    print()
    
    # ─────────────────────────────────────────────────────────
    # 2. ARTICLES PER SOURCE (Top 15)
    # ─────────────────────────────────────────────────────────
    print("📰 TOP 15 SOURCES BY ARTICLE COUNT")
    print("─" * 50)
    source_counts = df['source'].value_counts().head(15)
    for i, (source, count) in enumerate(source_counts.items(), 1):
        bar = "█" * min(count, 30)
        print(f"   {i:2}. {source:<30} {count:3} {bar}")
    print()
    
    # ─────────────────────────────────────────────────────────
    # 3. ARTICLES BY METHOD
    # ─────────────────────────────────────────────────────────
    print("📡 ARTICLES BY SCRAPING METHOD")
    print("─" * 50)
    method_counts = df['method'].value_counts()
    total = len(df)
    for method, count in method_counts.items():
        icon = get_method_icon(method)
        pct = 100 * count / total
        bar = "█" * int(pct / 2)
        print(f"   {icon} {method:<10} {count:4} ({pct:5.1f}%) {bar}")
    print()
    
    # ─────────────────────────────────────────────────────────
    # 4. KEYWORD DISTRIBUTION
    # ─────────────────────────────────────────────────────────
    print("🔑 KEYWORD DISTRIBUTION")
    print("─" * 50)
    # Flatten matched keywords
    all_keywords_found = []
    for kw_str in df['matched_keywords'].dropna():
        all_keywords_found.extend([k.strip() for k in str(kw_str).split(',') if k.strip()])
    
    keyword_counts = Counter(all_keywords_found)
    for kw, count in keyword_counts.most_common(10):
        pct = 100 * count / len(df)
        bar = "█" * int(pct / 2)
        print(f"   {kw:<15} {count:4} ({pct:5.1f}%) {bar}")
    print()
    
    # ─────────────────────────────────────────────────────────
    # 5. CONTENT LENGTH ANALYSIS
    # ─────────────────────────────────────────────────────────
    print("📏 CONTENT LENGTH ANALYSIS")
    print("─" * 50)
    df['content_length'] = df['full_content'].apply(lambda x: len(str(x)) if x else 0)
    df['word_count'] = df['full_content'].apply(lambda x: len(str(x).split()) if x else 0)
    
    print(f"   Average content length: {df['content_length'].mean():,.0f} characters")
    print(f"   Average word count: {df['word_count'].mean():,.0f} words")
    print(f"   Shortest article: {df['content_length'].min():,} chars")
    print(f"   Longest article: {df['content_length'].max():,} chars")
    print()
    
    # Content quality breakdown
    full_articles = (df['content_length'] >= 500).sum()
    partial_articles = ((df['content_length'] >= 200) & (df['content_length'] < 500)).sum()
    short_articles = (df['content_length'] < 200).sum()
    
    print("   Content quality:")
    print(f"   ✅ Full (≥500 chars): {full_articles} ({100*full_articles/len(df):.1f}%)")
    print(f"   ⚠️ Partial (200-500): {partial_articles} ({100*partial_articles/len(df):.1f}%)")
    print(f"   ❌ Short (<200): {short_articles} ({100*short_articles/len(df):.1f}%)")
    print()
    
    # ─────────────────────────────────────────────────────────
    # 6. AUTHOR ANALYSIS
    # ─────────────────────────────────────────────────────────
    print("✍️ AUTHOR ANALYSIS")
    print("─" * 50)
    has_author = (df['author'].notna() & (df['author'] != '')).sum()
    print(f"   Articles with author: {has_author} ({100*has_author/len(df):.1f}%)")
    
    top_authors = df[df['author'] != '']['author'].value_counts().head(5)
    if len(top_authors) > 0:
        print("   Top authors:")
        for author, count in top_authors.items():
            print(f"      • {author}: {count} articles")
    print()

print("=" * 70)

📊 ANALYTICS DASHBOARD

📈 BASIC STATISTICS
──────────────────────────────────────────────────
   Total articles: 394
   Unique sources: 36
   Date range:  to 2025-12-14T14:29:18+00:00

📰 TOP 15 SOURCES BY ARTICLE COUNT
──────────────────────────────────────────────────
    1. CNET                            32 ██████████████████████████████
    2. TechRadar                       26 ██████████████████████████
    3. AP Tech                         24 ████████████████████████
    4. Electrek                        22 ██████████████████████
    5. CleanTechnica                   20 ████████████████████
    6. Digital Trends                  19 ███████████████████
    7. Yahoo Finance Tech              18 ██████████████████
    8. TechCrunch                      18 ██████████████████
    9. IEEE Spectrum                   18 ██████████████████
   10. The AI Blog                     16 ████████████████
   11. CNBC Tech                       15 ███████████████
   12. ZDNet                    

In [10]:
# ============================================================
# CELL 10: SUMMARY REPORT & FAILURE DIAGNOSIS
# ============================================================
# Final summary and diagnosis of failed sources

print("=" * 70)
print("📋 FINAL SUMMARY REPORT")
print("=" * 70)
print()

# ─────────────────────────────────────────────────────────
# SUCCESS VS FAILURE BREAKDOWN
# ─────────────────────────────────────────────────────────
successful = [r for r in all_results if r["success"]]
failed = [r for r in all_results if not r["success"]]

print("✅ SUCCESSFUL SOURCES")
print("─" * 50)
print(f"   Count: {len(successful)}/{len(all_results)} ({100*len(successful)/len(all_results):.1f}%)")
print()

# Group by method used
method_groups = {}
for r in successful:
    method = r["method_used"]
    if method not in method_groups:
        method_groups[method] = []
    method_groups[method].append(r)

for method, results in sorted(method_groups.items(), key=lambda x: -len(x[1])):
    icon = get_method_icon(method)
    print(f"   {icon} {method.upper()} ({len(results)} sources):")
    for r in sorted(results, key=lambda x: -len(x["articles"]))[:5]:
        print(f"      • {r['source']}: {len(r['articles'])} articles ({r['total_elapsed']:.1f}s)")
    if len(results) > 5:
        print(f"      ... and {len(results)-5} more")
    print()

# ─────────────────────────────────────────────────────────
# FAILED SOURCES DIAGNOSIS
# ─────────────────────────────────────────────────────────
print("❌ FAILED SOURCES DIAGNOSIS")
print("─" * 50)
print(f"   Count: {len(failed)}/{len(all_results)} ({100*len(failed)/len(all_results):.1f}%)")
print()

if failed:
    # Group failures by reason
    failure_reasons = {}
    for r in failed:
        # Get the last error from methods tried
        errors = [v.get("error", "unknown") for v in r["methods_results"].values() if v.get("error")]
        reason = errors[-1] if errors else "unknown"
        
        # Simplify common reasons
        if "No RSS" in reason:
            reason = "No RSS feed"
        elif "API not" in reason:
            reason = "No API endpoint"
        elif "No sitemap" in reason:
            reason = "No sitemap"
        elif "No articles" in reason or "Empty" in reason:
            reason = "No articles found"
        elif "timeout" in reason.lower():
            reason = "Timeout"
        elif "403" in reason or "Access denied" in reason:
            reason = "Access denied (403)"
        elif "paywall" in reason.lower():
            reason = "Paywall"
        
        if reason not in failure_reasons:
            failure_reasons[reason] = []
        failure_reasons[reason].append(r["source"])
    
    # Print grouped failures
    for reason, sources in sorted(failure_reasons.items(), key=lambda x: -len(x[1])):
        print(f"   🔴 {reason} ({len(sources)} sources):")
        for src in sources[:5]:
            print(f"      • {src}")
        if len(sources) > 5:
            print(f"      ... and {len(sources)-5} more")
        print()

# ─────────────────────────────────────────────────────────
# PERFORMANCE METRICS
# ─────────────────────────────────────────────────────────
print("⚡ PERFORMANCE METRICS")
print("─" * 50)
total_elapsed = sum(r["total_elapsed"] for r in all_results)
avg_per_source = total_elapsed / len(all_results) if all_results else 0

print(f"   Total scraping time: {format_elapsed(total_elapsed)}")
print(f"   Average per source: {avg_per_source:.1f}s")
print(f"   Articles per minute: {60 * len(all_articles) / total_elapsed:.1f}" if total_elapsed > 0 else "   N/A")
print()

# Slowest sources
print("   🐢 Slowest sources:")
slowest = sorted(all_results, key=lambda x: -x["total_elapsed"])[:5]
for r in slowest:
    status = "✅" if r["success"] else "❌"
    print(f"      {status} {r['source']}: {r['total_elapsed']:.1f}s")
print()

# Fastest successful sources
print("   ⚡ Fastest successful sources:")
fastest = sorted([r for r in all_results if r["success"]], key=lambda x: x["total_elapsed"])[:5]
for r in fastest:
    print(f"      ✅ {r['source']}: {r['total_elapsed']:.1f}s ({len(r['articles'])} articles)")

print()
print("=" * 70)
print("🏁 SCRAPING SESSION COMPLETE")
print("=" * 70)

📋 FINAL SUMMARY REPORT

✅ SUCCESSFUL SOURCES
──────────────────────────────────────────────────
   Count: 36/50 (72.0%)

   📡 RSS (23 sources):
      • TechRadar: 26 articles (55.9s)
      • Electrek: 22 articles (9.5s)
      • CleanTechnica: 20 articles (8.7s)
      • Digital Trends: 19 articles (26.2s)
      • TechCrunch: 18 articles (8.2s)
      ... and 18 more

   🌐 HTML (9 sources):
      • CNET: 32 articles (52.1s)
      • Yahoo Finance Tech: 18 articles (94.5s)
      • CNBC Tech: 15 articles (64.2s)
      • Tom's Hardware: 12 articles (168.7s)
      • Google AI Blog: 4 articles (68.0s)
      ... and 4 more

   🏠 HOMEPAGE (4 sources):
      • AP Tech: 24 articles (79.2s)
      • The AI Blog: 16 articles (63.8s)
      • The Atlantic Tech: 13 articles (60.6s)
      • BBC Technology: 1 articles (95.3s)

❌ FAILED SOURCES DIAGNOSIS
──────────────────────────────────────────────────
   Count: 14/50 (28.0%)

   🔴 No articles found (11 sources):
      • OpenAI Blog
      • Marktechpost
 